# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic dataset metadata
print("Dataset Name: {}".format(dataset.metadata.name))
print("Description: {}".format(dataset.metadata.description))


## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by their @id
record_sets = dataset.metadata.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else ''}")

# Show the fields for each record set
for rs in record_sets:
    print(f"\nFields in record set @id: {rs.id}")
    for f in rs.fields:
        field_name = f.name if hasattr(f, 'name') else ''
        print(f"  - @id: {f.id}, name: {field_name}, type: {f.data_type if hasattr(f, 'data_type') else ''}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Gather all record set IDs
record_set_ids = [rs.id for rs in record_sets]

dataframes = dict()

for rs in record_sets:
    print(f"Loading records for record set {rs.id} ...")
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rs.id)))
        print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
        dataframes[rs.id] = df
    except Exception as e:
        print(f"Could not load records for {rs.id}: {e}")

# As an example, print a sample of the first record set loaded
if len(dataframes) > 0:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a record set to perform EDA (use first loaded if unsure)
rs_id = list(dataframes.keys())[0]
df = dataframes[rs_id].copy()
print(f"Analyzing record set: {rs_id}")

# Try to select a numeric field automatically
numeric_columns = df.select_dtypes(include=['float', 'int']).columns.tolist()
if len(numeric_columns) == 0:
    print("No numeric columns found.")
else:
    numeric_field = numeric_columns[0]
    print(f"Using '{numeric_field}' as the numeric field for analysis.")

    # Filter records where value is above a threshold
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} out of {len(df)} rows")
    display(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a categorical field if one is present
    categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
    group_field = categorical_columns[0] if len(categorical_columns) > 0 else None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No categorical columns to group by.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution if available
if len(numeric_columns) > 0:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in {rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Visualize group means if grouped_df is available and grouping field exists
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(12, 5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean of {numeric_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset using Croissant metadata, explored record sets and fields using their `@id`, extracted dataframes, performed exploratory analysis on numeric columns, and visualized basic data distributions. This pipeline can be modified for in-depth domain-specific analysis using the record set and field references.